In [5]:
import tangram as tg
import scanpy as sc
import numpy as np
import pandas as pd
import os

outdir = "/maiziezhou_lab2/yuling/MouseSpinal/label_transfer/Tangram/scNucleus_output"
data = sc.read_h5ad('/maiziezhou_lab2/yuling/datasets/obj_integrated_sc_nucleus.h5ad')
# reference
rna_path = '/maiziezhou_lab2/yuling/MERFISH_spinal_cord_resolved_0718.h5ad'
rna = sc.read_h5ad(rna_path)

reference_data = data
query_data = rna[rna.obs['Section ID'] == '0503_F4_C'].copy()
# Preprocessing
sc.pp.normalize_total(query_data)
sc.pp.log1p(query_data)

In [6]:
import numpy as np
from scipy.spatial import cKDTree

# ---- inputs ----
radius = 75  # same units as adata.obsm["spatial"]
coords = np.asarray(query_data.obsm["spatial"])

# If coords has extra dims (e.g., 3D), keep first 2 columns for (x,y)
if coords.ndim != 2:
    raise ValueError(f"adata.obsm['spatial'] should be 2D, got shape {coords.shape}")
if coords.shape[1] > 2:
    coords = coords[:, :2]

# ---- neighbor counting within radius ----
tree = cKDTree(coords)

# For each point, get indices of neighbors within radius (includes itself)
neighbors = tree.query_ball_point(coords, r=radius)

# Count neighbors excluding itself
neighbor_counts = np.fromiter((len(ixs) - 1 for ixs in neighbors), dtype=int, count=coords.shape[0])

# Store in adata.obs
query_data.obs[f"n_neighbors_r{int(radius)}"] = neighbor_counts

print(query_data.obs[f"n_neighbors_r{int(radius)}"].describe())


count    6602.000000
mean       33.102696
std        15.703753
min         2.000000
25%        21.000000
50%        30.000000
75%        42.000000
max        85.000000
Name: n_neighbors_r75, dtype: float64


In [7]:
query_data.obsm['spatial']

array([[8607.01041039, 1571.38068101],
       [8573.00979335, 1573.94455839],
       [8553.93255192, 1575.78204413],
       ...,
       [9621.19959363, 3264.61267101],
       [9649.24403753, 3268.4360139 ],
       [9641.09393852, 3271.66152148]])

In [8]:
sc.pp.normalize_total(reference_data)
sc.pp.log1p(reference_data)
sc.pp.highly_variable_genes(reference_data, n_top_genes=3000, flavor="seurat_v3")

In [9]:
import time
import resource
import sys

os.makedirs(outdir, exist_ok=True)
t0 = time.perf_counter()

hvg = reference_data.var_names[reference_data.var['highly_variable']].to_list()
genes = list(set(hvg).intersection(set(query_data.var_names)))
reference_data = reference_data[:, genes].copy()
query_data = query_data[:, genes].copy()
# Trim reference (Tangram is heavy on GPU/CPU memory)
#reference_data = reference_data[:25000].copy()

# Harmonize genes
tg.pp_adatas(reference_data, query_data, genes=None)
# Run Tangram mapping
tg_map = tg.map_cells_to_space(
    reference_data,
    query_data,
    density_prior='uniform',
    device='cuda'   # GPU if enough VRAM
)

# Project cell type annotation (make sure this obs column exists!)
tg.project_cell_annotations(
    adata_sp=query_data,
    adata_map=tg_map,
    annotation='final_cluster_assignment'
)

# Save results
df = pd.DataFrame(query_data.obsm['tangram_ct_pred'], index=query_data.obs_names)
outfile = os.path.join(outdir, "tangram_pred.csv")
df.to_csv(outfile)
print(f"Saved result: {outfile}")

elapsed_sec = time.perf_counter() - t0
rss_peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
if sys.platform == "darwin":
    peak_mib = rss_peak / (1024.0 ** 2)
else:
    peak_mib = rss_peak / 1024.0

metrics_path = os.path.join(outdir, "runtimeSec_memoryMiB.csv")
pd.DataFrame([{"Time": elapsed_sec, "Memory": peak_mib}]).to_csv(metrics_path, index=False)
print(f"Saved metrics: {metrics_path}")


INFO:root:135 training genes are saved in `uns``training_genes` of both single cell and spatial Anndatas.
INFO:root:135 overlapped genes are saved in `uns``overlap_genes` of both single cell and spatial Anndatas.
INFO:root:uniform based density prior is calculated and saved in `obs``uniform_density` of the spatial Anndata.
INFO:root:rna count based density prior is calculated and saved in `obs``rna_count_based_density` of the spatial Anndata.
INFO:root:Allocate tensors for mapping.


INFO:root:Begin training with 135 genes and uniform density_prior in cells mode...
INFO:root:Printing scores every 100 epochs.


Score: 0.280, KL reg: 0.000
Score: 0.940, KL reg: 0.002
Score: 0.953, KL reg: 0.002
Score: 0.956, KL reg: 0.001
Score: 0.957, KL reg: 0.001
Score: 0.958, KL reg: 0.001
Score: 0.958, KL reg: 0.001
Score: 0.959, KL reg: 0.001
Score: 0.959, KL reg: 0.001
Score: 0.959, KL reg: 0.001


INFO:root:Saving results..
INFO:root:spatial prediction dataframe is saved in `obsm` `tangram_ct_pred` of the spatial AnnData.


Saved result: /maiziezhou_lab2/yuling/MouseSpinal/label_transfer/Tangram/scNucleus_output/tangram_pred.csv
Saved metrics: /maiziezhou_lab2/yuling/MouseSpinal/label_transfer/Tangram/scNucleus_output/runtimeSec_memoryMiB.csv
